
# Documentación del Dataset de Pádel

Este notebook explica qué representa cada columna del dataset utilizado para análisis de jugadores de pádel mediante visión por computador.

El dataset fue generado usando:

1. Videos de partidos.
2. Etiquetado manual de keypoints en Roboflow.
3. Inferencia automática sobre videos.
4. Extracción de coordenadas y métricas a un dataset tabular.

---

## Estructura general

El dataset tiene **83 columnas** divididas en 4 grupos principales:

- Identificación
- Datos de posición por jugador
- Flags de detección sospechosa
- Visibilidad de jugadores



# Grupo 1 — Identificación (4 columnas)

Estas columnas indican de dónde viene cada fila del dataset.

No se usan como features del modelo.


In [ ]:

columnas_identificacion = {
    "cancha": "Cancha física donde se jugó el partido. Ejemplo: Cancha_1.",
    "partido": "Identificador del partido dentro de la cancha. Ejemplo: Partido_19.",
    "punto": "Identificador del punto dentro del partido. Un punto es un video desde el saque hasta el final del rally.",
    "frame": "Frame procesado dentro del video. Va de 3 en 3 porque el pipeline procesó 1 de cada 3 frames."
}

for k, v in columnas_identificacion.items():
    print(f"{k}: {v}")



# Grupo 2 — Datos de posición por jugador (72 columnas)

Cada jugador tiene un bloque de 18 columnas.

Jugadores:
- j1
- j2
- j3
- j4

Cada bloque contiene:

- confianza general del jugador
- centro del cuerpo
- keypoints anatómicos
- punta de la raqueta


In [ ]:

bloque_jugador = {
    "confianza": "Qué tan seguro está el modelo de que detectó correctamente al jugador.",
    "x_centro": "Posición horizontal del centro del cuerpo en píxeles.",
    "y_centro": "Posición vertical del centro del cuerpo en píxeles.",
    
    "cabeza_x": "Posición X de la cabeza.",
    "cabeza_y": "Posición Y de la cabeza.",
    "cabeza_confianza": "Confianza de detección de la cabeza.",
    
    "hombro_d_x": "Posición X del hombro derecho.",
    "hombro_d_y": "Posición Y del hombro derecho.",
    "hombro_d_confianza": "Confianza del hombro derecho.",
    
    "codo_x": "Posición X del codo.",
    "codo_y": "Posición Y del codo.",
    "codo_confianza": "Confianza del codo.",
    
    "muneca_x": "Posición X de la muñeca.",
    "muneca_y": "Posición Y de la muñeca.",
    "muneca_confianza": "Confianza de la muñeca.",
    
    "punta_raqueta_x": "Posición X de la punta de la raqueta.",
    "punta_raqueta_y": "Posición Y de la punta de la raqueta.",
    "punta_raqueta_confianza": "Confianza de la punta de la raqueta."
}

for k, v in bloque_jugador.items():
    print(f"{k}: {v}")



# Grupo 3 — Flags de detección sospechosa (4 columnas)

Columnas:

- j1_raqueta_sospechosa
- j2_raqueta_sospechosa
- j3_raqueta_sospechosa
- j4_raqueta_sospechosa

Estas columnas detectan errores del modelo.


In [ ]:

explicacion_sospechosa = '''
La flag vale True cuando:

distancia(muneca, punta_raqueta) < 5 píxeles

Eso es físicamente imposible porque la raqueta tiene longitud.

Cuando ocurre:
- el modelo confundió muñeca y punta de la raqueta
- ambos puntos aparecen superpuestos
- el siguiente frame puede generar un salto artificial
- eso puede producir falsos golpes

Estas flags permiten filtrar frames defectuosos.
'''

print(explicacion_sospechosa)



# Grupo 4 — Visibilidad de jugadores (3 columnas)

Columnas:

- j2_visible
- j3_visible
- j4_visible

Indican si el jugador fue detectado correctamente.


In [ ]:

visibilidad = {
    1: "Jugador detectado correctamente.",
    0: "Jugador no visible o no detectado."
}

for k, v in visibilidad.items():
    print(f"{k}: {v}")



# Resumen estructural

```text
cancha, partido, punto, frame
│
├── j1
│   ├── confianza
│   ├── centro del cuerpo
│   ├── cabeza
│   ├── hombro
│   ├── codo
│   ├── muñeca
│   └── punta de raqueta
│
├── j2
├── j3
├── j4
│
├── flags de detección sospechosa
└── visibilidad
```



# Grupo 5 — Ángulos Biomecánicos Calculados (20 columnas)

Estas columnas fueron calculadas trigonométricamente (usando el arcotangente) a partir de las posiciones relativas de los keypoints de cada jugador. Sirven como variables de ingeniería de características (feature engineering) óptimas para los modelos de Machine Learning.

Para cada uno de los 4 jugadores (`j1`, `j2`, `j3`, `j4`) se añadieron 5 ángulos en grados:
- `_angulo_codo_d`: Ángulo interior formado entre la muñeca, el codo y el hombro derecho.
- `_angulo_hombro_d`: Ángulo de apertura del brazo formado entre el codo, el hombro derecho y la cabeza.
- `_angulo_inclinacion_raqueta_d`: Ángulo de inclinación del vector de la raqueta en el plano del frame.
- `_angulo_raqueta_hombro_d`: Ángulo relativo entre la punta de la raqueta y la línea del hombro.
- `_angulo_muneca_cabeza_d`: Ángulo de posición relativa de la muñeca respecto a la altura de la cabeza.

# Grupo 6 — Categorización Automática de Golpeo (4 columnas)

Columnas de variable objetivo (Target): `j1_Y_golpe`, `j2_Y_golpe`, `j3_Y_golpe`, `j4_Y_golpe`.

Para etiquetar si en un frame ocurrió un golpeo de **Drive (Derecha)**, se evaluaron simultáneamente rangos cinemáticos teóricos basados en la literatura biomecánica del deporte, aplicando un margen de tolerancia de **±5 grados**.

### Criterio algorítmico de clasificación:
Un frame se categoriza como **Golpeo (1.0)** si cumple todas estas condiciones:
1. **Codo:** Extensión o apertura controlada entre **$100^\circ$ y $180^\circ$**.
2. **Hombro:** Brazo separado del torso en un rango entre **$90^\circ$ y $180^\circ$**.
3. **Inclinación Raqueta:** Ángulo de ataque de la raqueta entre **$60^\circ$ y $180^\circ$**.

*Nota: Si el jugador no se encuentra visible en el frame, el valor asignado es un dato faltante (`NaN`), y si no cumple los rangos anteriores se registra como **No Golpeo (0.0)**.*

In [ ]:
import numpy as np
import pandas as pd

# Definición de reglas teóricas usadas para generar el dataset final
DRIVE_RANGES = {
    "codo": (100.0, 180.0),
    "hombro": (90.0, 180.0),
    "inclinacion_raqueta": (60.0, 180.0)
}
MARGEN = 5.0  # Margen de tolerancia en grados

def explicar_logica_etiquetado(row, jugador):
    """
    Función que replica la lógica con la que se creó el dataset final.
    Verifica si los ángulos calculados están dentro del rango biomecánico del Drive.
    """
    col_codo = f"{jugador}_angulo_codo_d"
    col_hombro = f"{jugador}_angulo_hombro_d"
    col_raqueta = f"{jugador}_angulo_inclinacion_raqueta_d"
    
    # Validar si existen datos para el jugador en este frame
    if pd.isna(row[col_codo]) or pd.isna(row[col_hombro]):
        return np.nan
        
    # Verificar límites con margen de tolerancia
    codo_ok = (DRIVE_RANGES["codo"][0] - MARGEN) <= row[col_codo] <= (DRIVE_RANGES["codo"][1] + MARGEN)
    hombro_ok = (DRIVE_RANGES["hombro"][0] - MARGEN) <= row[col_hombro] <= (DRIVE_RANGES["hombro"][1] + MARGEN)
    raqueta_ok = (DRIVE_RANGES["inclinacion_raqueta"][0] - MARGEN) <= row[col_raqueta] <= (DRIVE_RANGES["inclinacion_raqueta"][1] + MARGEN)
    
    return 1.0 if (codo_ok and hombro_ok and raqueta_ok) else 0.0

print("Lógica algorítmica integrada correctamente para la documentación.")